In [ ]:
# import dependencies
import os
import numpy as np
import pandas as pd
import scanpy as sc
import loompy as lp
from MulticoreTSNE import MulticoreTSNE as TSNE
import json
import base64
import zlib
from pyscenic.plotting import plot_binarization
from pyscenic.export import add_scenic_metadata
from pyscenic.cli.utils import load_signatures
import matplotlib as mpl
import matplotlib.pyplot as plt
from scanpy.plotting import scatter
import seaborn as sns

In [4]:
os.chdir("/analysis")

# Load and Process Objects

In [5]:
# use counts data 
metacell_car_adata = sc.read_h5ad("./seurat_objects/Metacell_TNK_CAR.h5ad")
metacell_noncar_adata =  sc.read_h5ad("./seurat_objects/Metacell_TNK_NonCAR.h5ad")
metacell_mono_adata = sc.read_h5ad("./seurat_objects/Metacell_mono.h5ad")

In [15]:
metacell_mono_adata

AnnData object with n_obs × n_vars = 20455 × 25398
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'cells_merged', 'celltype_sample', 'new_celltype'
    var: 'features'
    obsm: 'X_harmony', 'X_umap'
    varm: 'HARMONY'

In [16]:
# create basic row and column attributes for the loom file:
row_attrs = {
    "Gene": np.array(metacell_car_adata.var_names) ,
}
col_attrs = {
    "CellID": np.array(metacell_car_adata.obs_names) ,
    "nGene": np.array( np.sum(metacell_car_adata.X.transpose() > 0 , axis=0)).flatten() ,
    "nUMI": np.array( np.sum(metacell_car_adata.X.transpose() , axis=0)).flatten() ,
}

row_attrs_noncar = {
    "Gene": np.array(metacell_noncar_adata.var_names) ,
}
col_attrs_noncar = {
    "CellID": np.array(metacell_noncar_adata.obs_names) ,
    "nGene": np.array( np.sum(metacell_noncar_adata.X.transpose() > 0 , axis=0)).flatten() ,
    "nUMI": np.array( np.sum(metacell_noncar_adata.X.transpose() , axis=0)).flatten() ,
}

row_attrs_mono = {
    "Gene": np.array(metacell_mono_adata.var_names) ,
}
col_attrs_mono = {
    "CellID": np.array(metacell_mono_adata.obs_names) ,
    "nGene": np.array( np.sum(metacell_mono_adata.X.transpose() > 0 , axis=0)).flatten() ,
    "nUMI": np.array( np.sum(metacell_mono_adata.X.transpose() , axis=0)).flatten() ,
}

In [17]:
lp.create("./pyscenic/metacell_CAR_loom.loom", metacell_car_adata.X.transpose(), row_attrs, col_attrs)
lp.create("./pyscenic/metacell_NonCAR_loom.loom", metacell_noncar_adata.X.transpose(), row_attrs_noncar, col_attrs_noncar)
lp.create("./pyscenic/metacell_Mono_loom.loom", metacell_mono_adata.X.transpose(), row_attrs_mono, col_attrs_mono)

In [18]:
lf = lp.connect( "./pyscenic/metacell_NonCAR_loom.loom", mode='r+', validate=False )

In [19]:
lf.close()